# How long can MoM-BH\*-1 last? A physics simulator (stage 3)

**The question:** stage 2 showed how MoM-BH\*-1 can hold itself together *right now*. But the black hole keeps eating its envelope, and once it holds more than about 10% of the total mass, no balanced envelope can exist (stage 2, Step 5a). So how long does a black hole star last, where is MoM-BH\*-1 in its life, and how can objects that live so briefly be common enough for JWST to find?

**Physics this stage adds:**
- **Feeding and light (E = mc²):** gas falls in, and a fraction ε = 0.1 of its rest-mass energy comes out as light, L = ε Ṁ_in c². The black hole keeps the rest, so it grows at **dM_BH/dt = L / (ε′c²)**, with ε′ = ε/(1 − ε) = 0.111.
- **Brightness:** the envelope shines at about its Eddington luminosity, **L = f × L_Edd(M_total)** with f ≈ 1. (Ball et al.'s full code gives f between 0.76 and 1.58; you'll test this.)
- **Energy conservation:** the light carries mass-energy away, so the total mass falls at **dM_total/dt = −L/c²**.
- **The end:** when the black hole reaches 10.15% of the total mass, there's no balanced envelope any more and the black hole star phase is over.

**Checks:** maths tests (Step 3); Ball et al.'s published growth history (Step 4); and two population tests using real surveys (Steps 6–7).

**How to use:** run the grey code cells from top to bottom with **Shift + Enter**. It all runs in a few seconds.

## Step 1 — Set up: constants, your stage 2 results, and Ball et al.'s growth history

This cell stores the constants, your results from stage 2 (luminosity, total mass, black-hole mass), and Ball et al.'s (2011) Table 1, which follows their 10,000-Sun quasi-star as its black hole grows.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# --- constants (cgs units) ---
G, c = 6.674e-8, 2.998e10
Msun, Lsun = 1.989e33, 3.828e33
YEAR, MYR = 3.156e7, 3.156e13
X = 0.7
KAPPA_ES = 0.2 * (1 + X)            # 0.34 cm^2/g, as in stage 2
Q_MAX = 0.1015                      # largest black-hole share with a balanced envelope (stage 2, Step 5a)

def L_edd(M):
    return 4*np.pi*G*c*M / KAPPA_ES

# --- your stage 2 results for MoM-BH*-1 ---
L_obs = 2.9e44                                   # erg/s
M_total_now = KAPPA_ES * L_obs / (4*np.pi*G*c)   # ~2 million Suns (stage 2, Step 2)
M_bh_now = 1e4 * Msun                            # best match (stage 2, Step 7)
M_bh_low, M_bh_high = 4.5e3*Msun, 1.8e4*Msun     # range across efficiencies (stage 2, Step 7)

# --- Ball et al. (2011) Table 1: their 10,000-Sun quasi-star growing ---
ball_t = np.array([0, 0.51, 1.03, 2.23, 3.70, 4.23])                    # time, Myr
ball_M = np.array([5, 100, 200, 500, 1000, 1194])                       # black hole, Suns
ball_L = np.array([3.48, 2.92, 3.40, 4.83, 6.05, 5.81]) * 1e8 * Lsun    # luminosity, erg/s
ball_T = np.array([14300, 5220, 4770, 4550, 4490, 4510])                # surface temperature, K

print(f"MoM-BH*-1 now: total mass {M_total_now/Msun:.2e} Suns, black hole {M_bh_now/Msun:,.0f} Suns "
      f"(share {M_bh_now/M_total_now:.4f}), luminosity {L_obs:.1e} erg/s")
print(f"Ball et al.'s run: L / L_Edd(10,000 Suns) = {np.round(ball_L/L_edd(1e4*Msun), 2)}")

# --- plot style and colours (same as stage 2) ---
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "axes.edgecolor": "#c3c2b7",
    "axes.labelcolor": "#0b0b0b", "axes.titlecolor": "#0b0b0b", "xtick.color": "#52514e",
    "ytick.color": "#52514e", "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2,
    "font.size": 11, "legend.frameon": False, "legend.fontsize": 9.5})
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
INK2, MUTED, ZONE = "#52514e", "#898781", "#ebeae5"

## Step 2 — The growth code

`grow()` follows the black hole and the total mass forward in time using the three rules above. It stops automatically when the black hole's share reaches the 10.15% limit.

`lifetime_formula()` is the same calculation done with pen and paper, ignoring the small mass carried off by light. Because both the growth rate and the Eddington luminosity are proportional to the total mass, the total mass cancels:

**lifetime ≈ (q_max − q_start) × ε′ × κc / (4πG f)**

So **every black hole star lives the same length of time, whatever its mass.** κc/(4πG) is about 385 million years; multiplied by 0.1015 × 0.111, that gives about 4.3 million years.

In [ ]:
def grow(M_total0, M_bh0, f=1.0, eps=0.1, q_max=Q_MAX, t_max_myr=2000):
    # follow the black hole as it eats the envelope; stop when its share reaches q_max
    eps_p = eps / (1 - eps)
    def rates(t, y):
        M_bh, M_tot = y
        L = f * L_edd(M_tot)
        return [L / (eps_p*c**2), -L / c**2]       # black hole grows; light carries mass away
    def limit(t, y):
        return y[0]/y[1] - q_max
    limit.terminal = True
    sol = solve_ivp(rates, [0, t_max_myr*MYR], [M_bh0, M_total0], events=limit,
                    rtol=1e-10, dense_output=True)
    t_end = sol.t_events[0][0] if sol.t_events[0].size else np.nan
    return sol, t_end

def lifetime_formula(q_start, f=1.0, eps=0.1, q_max=Q_MAX):
    eps_p = eps / (1 - eps)
    return (q_max - q_start) * eps_p * c * KAPPA_ES / (4*np.pi*G*f)

print(f"kappa c / (4 pi G) = {c*KAPPA_ES/(4*np.pi*G)/MYR:.0f} Myr")
print(f"Lifetime from a tiny seed (formula, f = 1, eps = 0.1): {lifetime_formula(0)/MYR:.2f} Myr")

## Step 3 — Layer 1: check the maths

Three tests, each printing **PASS** or **FAIL**:
1. **E = mc² bookkeeping.** For every gram the black hole gains, ε′ = 0.111 grams' worth of energy must leave as light. Check: (mass carried off by light) ÷ (black hole's gain) = 0.111.
2. **Code vs formula.** The computer's lifetime should match the pen-and-paper formula within about 2%. (The formula ignores the ~1% of mass carried off by light.)
3. **Mass doesn't matter.** Quasi-stars of 10,000, 1 million and 100 million Suns, all starting with a black hole at 0.05% of their mass, must live exactly as long as each other.

In [ ]:
def check(name, ok, detail):
    print(f"{'PASS' if ok else 'FAIL'}  {name}: {detail}")

# Test 1: E = mc^2 bookkeeping
sol, t_end = grow(2e6*Msun, 1e3*Msun)
M_bh_end, M_tot_end = sol.sol(t_end)
gain = M_bh_end - 1e3*Msun
radiated = 2e6*Msun - M_tot_end
check("E = mc^2 bookkeeping", abs(radiated/gain - 0.1/0.9) < 1e-6,
      f"light carried off {radiated/gain:.6f} grams per gram the black hole gained (expected {0.1/0.9:.6f})")

# Test 2: code vs formula
t_formula = lifetime_formula(1e3/2e6)
check("code matches formula", abs(t_end/t_formula - 1) < 0.02,
      f"code {t_end/MYR:.3f} Myr, formula {t_formula/MYR:.3f} Myr")

# Test 3: lifetime doesn't depend on mass
lives = [grow(M*Msun, 5e-4*M*Msun)[1] for M in [1e4, 1e6, 1e8]]
spread = (max(lives) - min(lives)) / np.mean(lives)
check("lifetime independent of mass", spread < 1e-6,
      "lifetimes " + ", ".join(f"{t/MYR:.4f}" for t in lives) + " Myr for 1e4, 1e6, 1e8 Suns")

## Step 4 — Layer 2: compare with Ball et al.'s growth history

Two checks against their full stellar-evolution run:

**4a. Same bookkeeping?** Feed their own luminosity history into your growth rule. If you're doing the E = mc² bookkeeping the same way they do, you'll get back their black-hole masses. **Expect agreement within about 1%.**

**4b. Does "shine at the Eddington limit" work?** Now let the model choose its own luminosity, L = f × L_Edd, and compare the black-hole masses at their times. Their luminosity varied between 0.76 and 1.58 times Eddington, so f = 1 is the middle guess, and f = 0.76 and 1.58 bracket it. **Expect f = 1 to match within about 25%.**

In [ ]:
# 4a: their luminosity, your bookkeeping
eps_p = 0.1 / 0.9
tt = np.linspace(0, ball_t[-1], 4001)
L_t = np.interp(tt, ball_t, ball_L)
gained = np.concatenate([[0], np.cumsum(0.5*(L_t[1:] + L_t[:-1]) * np.diff(tt) * MYR)]) / (eps_p*c**2)
M_from_their_L = 5 + gained / Msun
print("4a. Their luminosity -> black-hole mass (Suns):")
for t, Mb in zip(ball_t, ball_M):
    mine = np.interp(t, tt, M_from_their_L)
    print(f"   t = {t:4.2f} Myr: yours {mine:7.1f}, theirs {Mb:5.0f}  (ratio {mine/Mb:.3f})")

# 4b: your Eddington rule
print("\n4b. Eddington rule (L = f x L_Edd) -> black-hole mass (Suns) at their times:")
runs = {}
for f in [0.76, 1.0, 1.58]:
    sol, t_end = grow(1e4*Msun, 5*Msun, f=f)
    runs[f] = (sol, t_end)
    masses = [sol.sol(min(t*MYR, t_end))[0]/Msun for t in ball_t[:5]]
    print(f"   f = {f:4}: " + ", ".join(f"{m:6.0f}" for m in masses) + f"   | limit reached at {t_end/MYR:.2f} Myr")
print("   Ball:     " + ", ".join(f"{m:6.0f}" for m in ball_M[:5]) + "   | run ended at 4.23 Myr (share 0.119)")
print("   (a run that has already reached the limit keeps its final value)")

sol, t_end = runs[1.0]
t_grid = np.linspace(0, t_end, 400)
t_to_1000 = t_grid[np.argmax(sol.sol(t_grid)[0] >= 1000*Msun)]
print(f"\nTime to reach a black-hole share of 0.10 (1,000 Suns): yours {t_to_1000/MYR:.2f} Myr, Ball's 3.70 Myr")

fig, ax = plt.subplots(figsize=(8.5, 5))
t_plot = np.linspace(0, 5.8, 300)
lo_run, hi_run = runs[0.76], runs[1.58]
M_lo = [lo_run[0].sol(min(t*MYR, lo_run[1]))[0]/Msun for t in t_plot]
M_hi = [hi_run[0].sol(min(t*MYR, hi_run[1]))[0]/Msun for t in t_plot]
ax.fill_between(t_plot, M_lo, M_hi, color=BLUE, alpha=0.18, lw=0, label="Eddington rule, f = 0.76–1.58")
t_mid = np.linspace(0, runs[1.0][1], 300)
ax.plot(t_mid/MYR, runs[1.0][0].sol(t_mid)[0]/Msun, color=BLUE, label="Eddington rule, f = 1")
ax.plot(tt, M_from_their_L, color=ORANGE, ls="--", label="their luminosity, your bookkeeping")
ax.plot(ball_t, ball_M, "o", color=INK2, ms=8, label="Ball et al. 2011 (full code)")
ax.axhline(Q_MAX*1e4, color=MUTED, lw=1, ls=":")
ax.text(0.1, Q_MAX*1e4*1.04, "10% limit (1,015 Suns)", color=INK2, fontsize=9.5)
ax.set_xlim(0, 5.8); ax.set_ylim(0, 1400)
ax.set_xlabel("Time (million years)")
ax.set_ylabel("Black-hole mass (Suns)")
ax.set_title("Your growth rule follows Ball et al.'s full model", loc="left")
ax.legend(loc="lower right")
plt.tight_layout(); fig.savefig("s3_step4_vs_ball.png", dpi=150); plt.show()

**What it means:**
- **4a:** your bookkeeping matches theirs almost exactly, so you're using the same physics for how light and mass relate.
- **4b:** with f = 1, your black hole reaches the 10% limit at about 4.3 million years; their full code ended at 4.23 million years. Your growth starts a little faster and finishes a little slower, because their luminosity rose from 0.76 to about 1.5 times Eddington as the black hole grew.

So the simple rule "shine at the Eddington limit" reproduces the lifetime of a full stellar-evolution model within about 15%. One caution for later: Ball et al. also found that bigger quasi-stars burn through their envelopes a bit faster, so for a 2-million-Sun object like MoM-BH\*-1 the true lifetime may be somewhat shorter.

## Step 5 — MoM-BH\*-1's life story

Now run MoM-BH\*-1 from a small seed black hole (100 Suns; the starting size barely matters) inside 2 million Suns of envelope. The cell prints how old it is now, how long it has left, and how big its black hole will be at the end. It also shows how the lifetime depends on the two uncertain numbers, f and the efficiency ε, and what happens with Coughlin & Begelman's (2024) higher limit of about 60%.

**Expect:** a total life of about 4.3 million years, an age now of about 0.2 million years, and a black hole of about 200,000 Suns at the end.

In [ ]:
sol, t_end = grow(M_total_now, 100*Msun)
t_grid = np.linspace(0, t_end, 20001)
M_bh_t = sol.sol(t_grid)[0]
age = lambda M: np.interp(M, M_bh_t, t_grid)            # time for the black hole to reach mass M

print(f"Total life (f = 1, eps = 0.1):  {t_end/MYR:.2f} million years")
print(f"Age now (black hole {M_bh_now/Msun:,.0f} Suns): {age(M_bh_now)/MYR:.2f} million years "
      f"(range {age(M_bh_low)/MYR:.2f}-{age(M_bh_high)/MYR:.2f} for {M_bh_low/Msun:,.0f}-{M_bh_high/Msun:,.0f} Suns)")
print(f"Share of its life used so far: {age(M_bh_now)/t_end*100:.0f}%")
print(f"Time left: {(t_end - age(M_bh_now))/MYR:.2f} million years")
M_bh_end, M_tot_end = sol.sol(t_end)
print(f"Black hole at the end: {M_bh_end/Msun:,.0f} Suns")
print(f"Mass carried off as light over its whole life: {(M_total_now - M_tot_end)/Msun:,.0f} Suns")
print(f"Growth rate now: {L_obs/(0.1/0.9*c**2)*YEAR/Msun:.3f} Suns per year")

print("\nHow the lifetime depends on the uncertain numbers (formula, starting from a tiny seed):")
for f in [0.76, 1.0, 1.58]:
    print(f"   f = {f:4}, eps = 0.1  -> {lifetime_formula(0, f=f)/MYR:5.1f} Myr")
for eps in [0.05, 0.2, 0.3]:
    print(f"   f = 1.0,  eps = {eps:4} -> {lifetime_formula(0, eps=eps)/MYR:5.1f} Myr")
print(f"   Coughlin & Begelman limit (60% instead of 10%), same growth rule -> "
      f"{lifetime_formula(0, q_max=0.6)/MYR:.0f} Myr")

fig, ax = plt.subplots(figsize=(8.5, 5))
for f, ls in [(0.76, ":"), (1.58, ":")]:
    s, te = grow(M_total_now, 100*Msun, f=f)
    tg = np.linspace(0, te, 500)
    ax.plot(tg/MYR, s.sol(tg)[0]/Msun, color=BLUE, lw=1.2, ls=ls)
ax.plot(t_grid/MYR, M_bh_t/Msun, color=BLUE, label="black hole (f = 1; dotted: f = 0.76 and 1.58)")
ax.axhspan(Q_MAX*M_total_now/Msun, 1e7, color=ZONE, zorder=0)
ax.text(0.15, 3.2e5, "no balanced envelope: the black hole star phase is over", color=INK2, fontsize=10)
ax.errorbar(age(M_bh_now)/MYR, M_bh_now/Msun,
            yerr=[[(M_bh_now - M_bh_low)/Msun], [(M_bh_high - M_bh_now)/Msun]],
            fmt="o", color=ORANGE, ms=9, lw=2, capsize=4, label="MoM-BH*-1 now (stage 2)")
ax.set_yscale("log")
ax.set_xlim(0, 6); ax.set_ylim(80, 1e6)
ax.set_xlabel("Time since the black hole star formed (million years)")
ax.set_ylabel("Black-hole mass (Suns)")
ax.set_title("MoM-BH*-1 is early in a ~4-million-year life", loc="left")
ax.legend(loc="lower right")
plt.tight_layout(); fig.savefig("s3_step5_life_story.png", dpi=150); plt.show()

**What it means:** if MoM-BH\*-1 is a classic quasi-star, it's only about 0.2 million years into a life of about 4 million years: roughly the first 5%. Its brightness barely changes along the way, because it always shines near the Eddington limit of its (almost constant) total mass.

The lifetime depends on the assumptions: about 2.7–5.7 million years for f between 0.76 and 1.58, and 2–17 million years for efficiencies between 5% and 30%. With Coughlin & Begelman's higher limit, the black hole could keep growing for about 26 million years, if it kept growing the same way. In every case the phase is short: the universe was already about 660 million years old when we see MoM-BH\*-1.

## Step 6 — Population test 1: are hot little red dots young?

In Ball et al.'s run, a newborn quasi-star starts hot (14,300 K), cools to about 5,200 K within half a million years, then sits near 4,500 K for the rest of its life. That flat stretch is the "wall" quasi-stars can't cool past.

**The test:** if little red dots are quasi-stars caught at random moments in their lives, the share of them that look hotter than some temperature should match the share of a quasi-star's life spent that hot.

We use 5,000 K and 5,200 K. For these two thresholds the answer doesn't depend on exactly how fast the newborn cools, because Ball's table pins down when it passes them. The temperatures of the little red dots come from Curtis et al.'s (2026) fits, loaded from their public table.

In [ ]:
import pandas as pd

URL = "https://raw.githubusercontent.com/o-curtis/lrd-quasistar-fits/main/summary/photospheric_parameters.csv"
table = pd.read_csv(URL)
main = table[table["family"].isin(["primary", "supplementary/prism", "supplementary/desi",
                                   "supplementary/grating"])].drop_duplicates("target")
print(f"{len(main)} little red dots loaded")

# fraction of Ball's quasi-star life spent above each temperature (temperature falls steadily with time)
tt = np.linspace(0, ball_t[-1], 200001)
T_life = np.interp(tt, ball_t, ball_T)
thresholds = [5000, 5200]
life_share = [np.mean(T_life > T) * 100 for T in thresholds]
lrd_share = [np.mean(main.teff_50 > T) * 100 for T in thresholds]
lrd_low = [np.mean(main.teff_16 > T) * 100 for T in thresholds]      # if every fit is at its cool end
lrd_high = [np.mean(main.teff_84 > T) * 100 for T in thresholds]     # if every fit is at its hot end

for T, a, b, lo, hi in zip(thresholds, life_share, lrd_share, lrd_low, lrd_high):
    print(f"Hotter than {T} K: quasi-star spends {a:4.1f}% of its life; "
          f"little red dots: {b:4.1f}% (range {lo:.0f}-{hi:.0f}% across fit uncertainties)")
mom = main[main["target"] == "MoMBH1"].iloc[0]
print(f"\nMoM-BH*-1: {mom.teff_50:,.0f} K in Curtis et al.'s fit; hotter than "
      f"{np.mean(main.teff_50 < mom.teff_50)*100:.0f}% of the sample")

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.set_axisbelow(True)
x = np.arange(len(thresholds))
w = 0.36
ax.bar(x - w/2, life_share, w, color=AQUA, label="share of a quasi-star's life (Ball et al.)")
ax.bar(x + w/2, lrd_share, w, color=BLUE, label="share of little red dots (Curtis et al.)")
ax.errorbar(x + w/2, lrd_share, yerr=[np.subtract(lrd_share, lrd_low), np.subtract(lrd_high, lrd_share)],
            fmt="none", ecolor=INK2, capsize=4, lw=1.2)
for xi, a, b in zip(x, life_share, lrd_share):
    ax.text(xi - w/2, a + 0.8, f"{a:.0f}%", ha="center", color=INK2, fontsize=10)
    ax.text(xi + w/2 + 0.07, b + 0.8, f"{b:.0f}%", ha="left", color=INK2, fontsize=10)
ax.set_xticks(x)
ax.set_xticklabels([f"hotter than {T:,} K" for T in thresholds])
ax.set_ylabel("Percent")
ax.set_ylim(0, 32)
ax.set_title("Hot little red dots are about as rare as hot young quasi-stars", loc="left")
ax.legend(loc="upper right")
plt.tight_layout(); fig.savefig("s3_step6_hot_fraction.png", dpi=150); plt.show()

**What it means:** a quasi-star spends about 18% of its life hotter than 5,000 K and about 12% hotter than 5,200 K. Among the little red dots, about 14% and 9% are that hot. Within the uncertainties, these agree. This is what you'd expect if little red dots are quasi-stars caught at random moments in their lives.

MoM-BH\*-1 is one of the hottest objects in the sample, which fits Step 5: it would be a young black hole star, still cooling towards the wall.

**Caveats:** Ball's run is for a 10,000-Sun quasi-star, 200 times lighter than MoM-BH\*-1. Curtis et al.'s temperatures come from atmosphere models (they put MoM-BH\*-1 at about 6,500 K, while your blackbody gives about 5,000–5,500 K). And about half of the little red dots are cooler than Ball's 4,500 K wall. Heavier quasi-stars may have a slightly cooler wall, but that's for future work. Treat this as a consistency check, not a proof.

## Step 7 — Population test 2: how can they be so common?

A June 2026 study (arXiv:2606.30253) counted little red dots at least as bright as 3 × 10⁴⁴ erg/s, about MoM-BH\*-1's brightness. It found about **3.4 × 10⁻⁵ per Mpc³** (range 1.0–9.0 × 10⁻⁵), and no significant change between z ≈ 7 and z ≈ 2.4. Earlier surveys (Kocevski et al. 2025; Ma et al. 2025) instead found their numbers dropping below z ≈ 4.5. So we try two eras:
- **A:** common from z ≈ 9 to z ≈ 4.5 (the earlier surveys);
- **B:** common from z ≈ 9 to z ≈ 2.4 (the June 2026 study).

If each black hole star lasts only ~4.3 million years, new ones must keep forming to keep that many on the sky. The cell works out how many generations that takes and how many must form in total.

It also works out how much black-hole mass they'd build. This uses only the number on the sky, the growth rate and the length of the era, not the lifetime. The result is compared with the mass in all black holes today, about 4.4–5.9 × 10⁵ Suns per Mpc³ (Graham & Driver 2007).

In [ ]:
def cosmic_age_gyr(z, H0=67.7, Om=0.31):
    # age of a flat universe with matter + dark energy, in billions of years
    OL = 1 - Om
    t_hubble = 977.8 / H0
    return 2*t_hubble/(3*np.sqrt(OL)) * np.arcsinh(np.sqrt(OL/Om) * (1 + z)**-1.5)

n_seen, n_low, n_high = 3.4e-5, 1.0e-5, 9.0e-5           # per Mpc^3, L >= 3e44 erg/s
life_myr = lifetime_formula(0) / MYR                     # ~4.3 Myr
growth = 3e44 / (0.1/0.9*c**2) * YEAR / Msun             # Suns per year, at 3e44 erg/s
rho_bh_today = (4.4e5, 5.9e5)                            # Suns per Mpc^3

print(f"Age of the universe at MoM-BH*-1's redshift (7.76): {cosmic_age_gyr(7.76)*1000:.0f} Myr")
print(f"Lifetime of each black hole star: {life_myr:.1f} Myr\n")
eras = {"A (z 9 -> 4.5)": (9, 4.5), "B (z 9 -> 2.4)": (9, 2.4)}
built = {}
for name, (z1, z2) in eras.items():
    length = (cosmic_age_gyr(z2) - cosmic_age_gyr(z1)) * 1000          # Myr
    generations = length / life_myr
    formed = n_seen * generations
    built[name] = n_seen * growth * length * 1e6                         # Suns per Mpc^3
    print(f"Era {name}: lasts {length:,.0f} Myr")
    print(f"   generations needed:        {generations:,.0f}")
    print(f"   black hole stars formed:   {formed:.1e} per Mpc^3 (range {n_low*generations:.1e}-{n_high*generations:.1e})"
          f"  = one per {1/formed:,.0f} Mpc^3")
    print(f"   black-hole mass they build: {built[name]:,.0f} Suns per Mpc^3 = "
          f"{built[name]/rho_bh_today[1]*100:.1f}-{built[name]/rho_bh_today[0]*100:.1f}% of today's\n")
print(f"New black hole stars needed: {n_seen/life_myr:.1e} per Mpc^3 every million years")

fig, ax = plt.subplots(figsize=(7.5, 4.3))
ax.set_axisbelow(True)
labels =["built by black hole stars,\nera A", "built by black hole stars,\nera B", "in all black holes\ntoday"]
values = [built["A (z 9 -> 4.5)"], built["B (z 9 -> 2.4)"], np.mean(rho_bh_today)]
ax.bar(labels, values, color=[BLUE, BLUE, AQUA], width=0.55)
for i, v in enumerate(values):
    ax.text(i, v*1.25, f"{v:,.0f}", ha="center", color=INK2, fontsize=10)
ax.set_yscale("log"); ax.set_ylim(100, 3e6)
ax.set_ylabel("Black-hole mass (Suns per Mpc³)")
ax.set_title("Black hole stars build under 1% of today's black-hole mass", loc="left")
plt.tight_layout(); fig.savefig("s3_step7_how_common.png", dpi=150); plt.show()

**What it means:**
- **They're common because they keep being reborn.** Each one lasts only ~4 million years, so to keep about 3 × 10⁻⁵ per Mpc³ on the sky, roughly 8 new ones must form per million Mpc³ every million years. Over the era that's about 180–500 generations: 6 × 10⁻³ to 2 × 10⁻² per Mpc³ in total, about one for every 60–160 Mpc³ of space.
- **They build only a small share of today's black-hole mass:** about 0.2–0.8%. If black hole stars are real, they could be how many black holes *start* (as seeds of up to ~200,000 Suns), while most black-hole mass is gained later, by ordinary feeding over billions of years.

**Caveats:** this assumes little red dots are black hole stars, counts only those at least as bright as MoM-BH\*-1, and uses a single growth rate. The number on the sky is uncertain by a factor of about 3, and the two surveys disagree on how long the era lasted.

# What stage 3 tells you

- **A black hole star lasts about 4 million years** (2–17 million, depending on f and the efficiency), **whatever its mass**, because the growth rate and the brightness both scale with the total mass. Your simple growth rule reproduces Ball et al.'s full model to within about 15% (Steps 3–4).
- **MoM-BH\*-1 would be about 0.2 million years old**, about 5% into its life, with a black hole that will grow to about 200,000 Suns before the phase ends (Step 5).
- **Hot little red dots are about as rare as hot young quasi-stars** (roughly 9–14% vs 12–18%), and MoM-BH\*-1 is one of the hottest, consistent with it being young (Step 6).
- **They're common because new ones keep forming:** about 180–500 generations over the little-red-dot era. Together they'd build only about 0.2–0.8% of today's black-hole mass, so they'd be how black holes start, not where most of their mass comes from (Step 7).

**For a write-up, cite:** Ball et al. 2011 (MNRAS 414, 2751); Coughlin & Begelman 2024 (ApJ 970, 158); Curtis et al. 2026 (arXiv:2609.09265); arXiv:2606.30253 for the little-red-dot numbers; Kocevski et al. 2025 and Ma et al. 2025 for the earlier decline; Graham & Driver 2007 (MNRAS 380, L15) for today's black-hole mass density.

**Next:** stage 4 (AI) would run stages 2 and 3 many thousands of times with different f, ε, masses and limits, train a neural network to imitate them, and find every combination that fits all your measurements at once, with honest error bars.